# Hansen label-noise audit (Kaggle)
Runs the same pipeline as `scripts/hansen_audit/` with more seeds. **Not executed by the author on Kaggle**; the CPU run in `results/` is the only run reported.

Setup: (1) add the dataset `rajulkabir/eudr-satellite` as input; (2) upload this repo's `hansen_audit/` and `scripts/hansen_audit/` folders as a dataset named `hansen-audit-code`; (3) Accelerator can stay off, the model is ~20k parameters; GPU only matters if you extend to the full Siamese DeepLab (see last cell).

In [ ]:
import glob, os, subprocess, sys
DATA = os.path.dirname(glob.glob('/kaggle/input/**/farms_osm.csv', recursive=True)[0])
CODE = os.path.dirname(glob.glob('/kaggle/input/**/hansen_audit/__init__.py', recursive=True)[0])
print(DATA, CODE)
os.makedirs('/kaggle/working/results', exist_ok=True)
sys.path.insert(0, CODE)

In [ ]:
subprocess.run([sys.executable, f'{CODE}/scripts/hansen_audit/build_features.py', DATA, '/kaggle/working/results/features.npz'], check=True)

In [ ]:
# SEEDS=20 as a robustness run; reference metrics optional ('none' skips experiment D)
subprocess.run([sys.executable, f'{CODE}/scripts/hansen_audit/run_experiments.py', '/kaggle/working/results/features.npz', 'none', '/kaggle/working/results/experiments_kaggle.json', '15', '20'], check=True)

In [ ]:
import json; print(json.dumps(json.load(open('/kaggle/working/results/experiments_kaggle.json')), indent=1)[:6000])

## Optional GPU extension (designed, not implemented here)
Replace `torch.nn.functional.cross_entropy` in `src/change_siamese_train.py` with `hansen_audit.losses.gce` (q=0.7) and compare mIoU against Hansen labels with the same farm-grouped split. Report as agreement with Hansen, per PROTOCOL.md.